# Data Querying

`example_data.db` holds decoded AIS traffic, but a database is only useful once you can pull the slice you need out of it. This notebook covers `DBQuery`, the class that turns time ranges, bounding boxes, and MMSI filters into SQL, and `TrackGen`, which turns the resulting rows into per-vessel track dictionaries ready for analysis or plotting.

**What you will learn**
- How `DBQuery` combines a database connection, a time range, and a callback into a query
- The difference between the `in_timerange_validmmsi`, `in_time_bbox_validmmsi`, and related callbacks in `aisdb.database.sqlfcn_callbacks`
- How to build a bounding box with `DomainFromPoints` and query inside it
- How `gen_qry` streams grouped rows and how `TrackGen` converts them into tracks
- What keys a track dictionary carries and how to inspect one without opening a map

Companion GitBook page: [Data Querying](https://aisviz.gitbook.io/documentation/tutorials/data-querying).

In [ ]:
# %pip install aisdb
# Running on Google Colab? Uncomment the line above to install AISdb before continuing.

# %pip install nest_asyncio
# nest_asyncio lets aisdb.web_interface.visualize() run its own event loop inside a notebook kernel.

## Setup

This notebook queries `example_data.db`, the bundled Gulf of Maine dataset also used in [`1-database-loading.ipynb`](1-database-loading.ipynb). `nest_asyncio.apply()` is only needed once, before the guarded `visualize()` call near the end.

In [ ]:
from datetime import datetime

import aisdb
from aisdb import DomainFromPoints
import nest_asyncio

nest_asyncio.apply()

dbpath = './example_data.db'

## Querying within a time range

`DBQuery` takes a connection object, never a connection and a `dbpath` together, plus keyword arguments describing the query. The `callback` argument points at a function from `aisdb.database.sqlfcn_callbacks`, AISdb's set of ready-made SQL filters; `sqlfcn_callbacks.in_timerange_validmmsi` restricts rows to `start`/`end` and drops rows with an invalid MMSI in the same pass.

`gen_qry` runs the query lazily. It does not hand back one AIS position at a time, it yields a batch of rows per vessel, sorted by MMSI then by time, which is exactly the grouping `TrackGen` expects next. `TrackGen` consumes that generator and yields one dictionary per vessel, so pass `gen_qry`'s output straight through rather than iterating it yourself.

In [ ]:
start_time = datetime.strptime('2022-01-01 00:00:00', '%Y-%m-%d %H:%M:%S')
end_time = datetime.strptime('2022-01-08 00:00:00', '%Y-%m-%d %H:%M:%S')

with aisdb.SQLiteDBConn(dbpath=dbpath) as dbconn:
    qry = aisdb.DBQuery(
        dbconn=dbconn,
        start=start_time,
        end=end_time,
        callback=aisdb.database.sqlfcn_callbacks.in_timerange_validmmsi,
    )
    rowgen = qry.gen_qry()
    tracks_timerange = list(aisdb.track_gen.TrackGen(rowgen, decimate=False))

print(f'{len(tracks_timerange)} tracks between {start_time.date()} and {end_time.date()}')

### Inspecting a track

Each track is a plain dictionary. Static fields such as `mmsi` are scalars, dynamic fields such as `lon`, `lat`, `time`, and `sog` are NumPy arrays with one entry per position report. Printing the keys and the first couple of values is a quick way to confirm a query returned what you expected before reaching for a map.

In [ ]:
first_track = tracks_timerange[0]
print('keys:', sorted(first_track.keys()))
print('mmsi:', first_track['mmsi'])
print('first 3 lon/lat/sog:')
for lon, lat, sog in zip(first_track['lon'][:3], first_track['lat'][:3], first_track['sog'][:3]):
    print(f'  lon={lon:.5f}, lat={lat:.5f}, sog={sog:.2f}')

## Callback functions

A callback is a small function in `aisdb.database.sqlfcn_callbacks` that returns the SQL `WHERE` clause `DBQuery` appends to its query, built from whatever keyword arguments you pass in (`start`, `end`, `xmin`, `xmax`, `ymin`, `ymax`, `mmsis`, and so on). Swapping the callback is how you change what a query filters on without touching the rest of the `DBQuery` call. `in_timerange_validmmsi` above only looks at `start` and `end`; `in_time_bbox_validmmsi` used next also looks at the bounding box arguments, so it needs `xmin`/`xmax`/`ymin`/`ymax` supplied alongside the time range. Other callbacks worth knowing about include `in_validmmsi_bbox` (bbox and valid MMSI, no time filter) and `in_timerange_inmmsi` (time range plus an explicit MMSI list). Pick the callback whose name matches the filters you actually need; passing arguments a callback does not use is harmless, but omitting one it does use raises a SQL error at query time.

## Querying within a bounding box

`DomainFromPoints` builds a bounding box from a center point and a radius in meters, which is usually easier than hand-picking longitude and latitude bounds. The point below sits mid-channel in the Gulf of Maine, with an 80 km radius wide enough to catch several vessels in the sample week. Combining the resulting `domain.boundary` values with `in_time_bbox_validmmsi` filters on both the time window and the box in a single query.

In [ ]:
domain = DomainFromPoints(points=[(-69.8, 43.2)], radial_distances=[80000])

with aisdb.SQLiteDBConn(dbpath=dbpath) as dbconn:
    qry = aisdb.DBQuery(
        dbconn=dbconn,
        start=start_time,
        end=end_time,
        xmin=domain.boundary['xmin'],
        xmax=domain.boundary['xmax'],
        ymin=domain.boundary['ymin'],
        ymax=domain.boundary['ymax'],
        callback=aisdb.database.sqlfcn_callbacks.in_time_bbox_validmmsi,
    )
    rowgen = qry.gen_qry()
    tracks_bbox = list(aisdb.track_gen.TrackGen(rowgen, decimate=False))

print(f'{len(tracks_bbox)} tracks inside the bounding box, {start_time.date()} to {end_time.date()}')

## Visualizing the results

`aisdb.web_interface.visualize` starts a local web server and opens a browser tab to render the tracks on a map, which does not work on a headless test host. Keep `SHOW_MAP` set to `False` here; flip it to `True` when running this notebook on your own machine to see both queries plotted, the bounding-box run also passes `domain` so its zone polygon is drawn.

In [ ]:
SHOW_MAP = False  # flip to True locally to open the interactive map in a browser

In [ ]:
if SHOW_MAP:
    aisdb.web_interface.visualize(
        tracks_timerange,
        visualearth=True,
        open_browser=True,
    )

    aisdb.web_interface.visualize(
        tracks_bbox,
        domain=domain,
        visualearth=True,
        open_browser=True,
    )

## Next steps

The tracks pulled here still carry the noise inherent to raw AIS: duplicate points, GPS jumps, and implausible speeds among them. Continue to [`3-data-cleaning.ipynb`](3-data-cleaning.ipynb) to denoise them, covered in more depth on the [Data Cleaning GitBook page](https://aisviz.gitbook.io/documentation/tutorials/data-cleaning).